# Tailor — Worked Example: a real strong-motion number, reproducibly

A 20-minute, end-to-end walkthrough for a researcher who has never used
Tailor. By the last cell you will have reproduced a **recognizable number
from a recognizable public dataset** — the peak ground acceleration of the
1994 Northridge earthquake at the Tarzana station — and seen it flow through
the same governed pipeline (tiered access, consent gate, audit log, vault
memory) that Tailor applies to *any* data source.

This is **not a demo on synthetic data**. It downloads a real US-government
public-domain record, pins it by SHA-256, records that provenance *before*
touching the bytes, and ends on a hard assertion cell that fails loudly if
the number is wrong.

**What you'll see**

1. **Setup** — a fingerprint-pinned runtime download (per [ADR 0042](../adr/0042-runtime-download-of-public-datasets-in-docs.md)), with the provenance block printed before any analysis.
2. **Tier 1** — `seismic_record_summary`: peak ground acceleration, Arias intensity, significant duration, response spectra — a server-computed scalar summary, no consent gate, no raw trace in context.
3. **Tier 2** — `seismic_downsampled`: the consent gate fires, you approve, and a downsampled trace comes back to plot.
4. **Vault round-trip** — an evidence note written to plain Obsidian-compatible markdown.
5. **Audit log** — the SQLite row the router wrote, asserting `scrubber_id` is present.
6. **Reproduction** — assert the raw uncorrected peak == **1.927 g (V1)**, and contrast it with the instrument-corrected **1.779 g (V2)** — the provenance lesson.

…then a short **breadth coda**: point the *same* pipeline at the Ames Housing
dataset (earthquakes → house prices) with zero new code.

**Setup.** From the repo root: `pip install -e ".[dev]"`, plus `pip install
jupyter matplotlib`. A network connection is needed for the two pinned
downloads; if it is unavailable the notebook falls back to cached expected
values so every cell still runs.


## 1. Setup — pinned download with provenance recorded first

Tailor exists to make data flows auditable. So the very first thing this
notebook does with external data is record *where it came from* and *verify
it is exactly the bytes we pinned* — before a single analytical cell runs.
The risk a reviewer would flag in any "download and analyze" notebook (an
uninstrumented data flow) is inverted into the opening demonstration.

In [ ]:
import csv
import datetime
import hashlib
import io
import json
import math
import sqlite3
import tempfile
import urllib.request
import zipfile
from pathlib import Path

# ── Pinned public datasets (ADR 0042: openly-licensed, anonymous GET,
#    fingerprint-pinned, provenance recorded before analysis) ──
SEISMIC_URL = (
    "https://www.strongmotioncenter.org/NCESMD/data/"
    "northridge_17jan1994/ce24436r.zip"
)
SEISMIC_SHA256 = "b7483d6a86602db1ac19337a4b55ee0cccd8e172a551a499a8f913f3d87ec57a"
SEISMIC_INNER = "TARZANA.RAW"
SEISMIC_LICENSE = (
    "U.S. Government public domain — CESMD / California Geological Survey "
    "Strong Motion Instrumentation Program (no login, anonymous GET)"
)

# The two headline numbers this notebook is about:
RAW_PGA_EXPECTED = 1.927   # raw uncorrected peak (V1)
V2_PGA = 1.779             # processed / instrument-corrected peak (V2)

AMES_URL = "https://jse.amstat.org/v19n3/decock/AmesHousing.txt"
AMES_SHA256 = "6cfe6cb525ba437de428653a1040e2aed7d696640bf75203786a6d7a0e67cfcc"
AMES_LICENSE = "Openly shared (De Cock 2011, J. Statistics Education) — no login"
AMES_ROWS_EXPECTED = 2930

# Everything lives in a tempdir — nothing touches your real config.
workdir = Path(tempfile.mkdtemp(prefix="tailor-seismic-"))
config_dir = workdir / "config"; config_dir.mkdir()
data_dir = workdir / "data"; data_dir.mkdir()
records_dir = workdir / "records"; records_dir.mkdir()
vault_dir = workdir / "vault"; vault_dir.mkdir()
print("Workspace:", workdir)

In [ ]:
def fetch_pinned(url, sha256, *, inner=None):
    """Download, verify SHA-256, optionally extract an inner zip member.

    Returns (bytes_or_None, provenance_dict). On a download failure the
    bytes are None and the caller falls back to a cached value, so the
    notebook still runs offline. A *checksum mismatch* is different: it
    raises loudly rather than silently analyzing drifted upstream bytes.
    """
    retrieved_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    prov = {"url": url, "sha256_expected": sha256, "retrieved_at": retrieved_at}
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "tailor-worked-example"})
        blob = urllib.request.urlopen(req, timeout=60).read()
    except Exception as exc:  # offline / network error -> graceful fallback
        prov["status"] = f"download unavailable ({type(exc).__name__}) — using cached value"
        return None, prov
    actual = hashlib.sha256(blob).hexdigest()
    if actual != sha256:
        raise AssertionError(
            f"SHA-256 mismatch for {url}\n  expected {sha256}\n  actual   {actual}\n"
            "Upstream bytes changed — refusing to analyze drifted data (ADR 0042)."
        )
    prov["status"] = "verified"
    if inner is not None:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(inner)
    return blob, prov

In [ ]:
raw_bytes, prov = fetch_pinned(SEISMIC_URL, SEISMIC_SHA256, inner=SEISMIC_INNER)

print("PROVENANCE  (recorded before any analysis — ADR 0042)")
print("  source URL   :", prov["url"])
print("  SHA-256      :", prov["sha256_expected"])
print("  license      :", SEISMIC_LICENSE)
print("  retrieved at :", prov["retrieved_at"])
print("  status       :", prov["status"])

In [ ]:
def write_fallback_record(path):
    """Synthetic stand-in used only when the pinned download is unavailable.

    A decaying sinusoid scaled to the documented raw uncorrected peak
    (1.927 g) so the notebook still runs offline and the reproduction
    assertion remains meaningful. Clearly labelled as a fallback.
    """
    n, dt = 400, 0.005
    acc = [1.927 * math.sin(2 * math.pi * 2 * i * dt) * math.exp(-0.4 * i * dt)
           for i in range(n)]
    acc[40] = 1.927  # pin the exact documented raw peak
    raw = [a * 10.0 for a in acc]            # g -> G/10
    times = [i * dt for i in range(n)]
    header = [
        "TARZANA - CEDAR HILL NURSERY  (SYNTHETIC FALLBACK, offline)",
        "COSMOS V1 UNCORRECTED ACCELERATION",
        "CHAN  1:  90 DEG",
        f"NO. OF POINTS =  {n}  RECORD LENGTH = {round((n - 1) * dt, 3)} SEC",
        "UNITS OF UNCOR ACCEL ARE SEC AND G/10.",
    ]
    fields = []
    for t, a in zip(times, raw, strict=True):
        fields.append(f"{t:7.3f}")
        fields.append(f"{a:7.3f}")
    lines = ["".join(fields[i:i + 10]) for i in range(0, len(fields), 10)]
    path.write_text("\n".join(header + lines) + "\n", encoding="utf-8")


record_path = records_dir / "TARZANA.RAW"
if raw_bytes is not None:
    record_path.write_bytes(raw_bytes)
    RECORD_SOURCE = "real CESMD download"
    print("saved", record_path.name, f"({len(raw_bytes):,} bytes) — {RECORD_SOURCE}")
else:
    write_fallback_record(record_path)
    RECORD_SOURCE = "synthetic fallback (offline)"
    print("wrote", record_path.name, f"— {RECORD_SOURCE}")

## 2. Tier 1 — the stage summary, no gate

Tier 1 is where most questions are answered. The router calls the
strong-motion child's server-side analytics and returns a compact summary —
**no raw acceleration trace enters the LLM's context**. The headline number
is the peak ground acceleration; the child also computes Arias intensity,
the 5–95 % significant duration, and the 5 %-damped response spectrum.

We wire the router exactly the way `tailor serve` does, minus the stdio
transport so we can read return values directly.

In [ ]:
from tailor.children.strong_motion import StrongMotionChild
from tailor.framework.router import RouterMCP
from tailor.framework.vault import VaultLayer, VaultWriter

# Tell the child where the records live.
(config_dir / "user_config.json").write_text(
    json.dumps({"strong_motion": {"path": str(records_dir)}}),
    encoding="utf-8",
)

router = RouterMCP(name="seismic-worked-example", data_dir=data_dir)
sm = StrongMotionChild(config_dir=config_dir, data_dir=data_dir)
router.register_child(sm)

# Framework-level vault layer (cross-session memory). It skips the
# biosensor-tier gates because its payload is analyst notes, not data.
vault_writer = VaultWriter(
    vault_path=vault_dir, data_dir=data_dir,
    vaultable_tools=set(sm.vaultable_tools), max_hr=185,
)
router.register_post_execute_hook(vault_writer)
router.register_vault_layer(VaultLayer(vault_path=vault_dir, vault_writer=vault_writer))


async def call(tool_name, params=None):
    """Dispatch a tool the way an MCP client would; return the parsed result."""
    chunks = await router._dispatch(tool_name, params or {})
    return json.loads(chunks[0].text)


print("registered domains:", router.registered_domains)
print("tool count        :", len(router.registered_tools))

In [ ]:
# entity_id is the station-event code — audit scoping only (ADR 0009).
ENTITY = "CE24436-TARZANA-CH1"

summary = await call(
    "seismic_record_summary", {"file_id": "TARZANA.RAW", "entity_id": ENTITY},
)

print("station :", summary["station"])
print("channel :", summary["channel"], " azimuth:", summary["azimuth"], "deg")
print("samples :", summary["n_samples"], " dt:", summary["dt_s"], "s")
print()
print(f"  RAW UNCORRECTED PEAK (V1)   = {summary['pga_g']:.3f} g")
print(f"  Arias intensity             = {summary['arias_intensity_ms']:.2f} m/s")
print(f"  significant duration 5-95%  = {summary['strong_motion_duration_s']:.2f} s")
print( "  Sa(T), 5% damping (g):")
for period, sa in summary["spectral_acceleration_g"].items():
    print(f"      {period:>8}  {sa:.3f}")
print()
print("  tier:", summary["_meta"]["tier"], "— server-computed scalar, no raw trace in context")

Note the **labelling**: 1.927 g is the *raw uncorrected peak (V1)* — the
maximum of the as-recorded accelerogram. It is not yet instrument-corrected;
Section 6 contrasts it with the processed V2 value. Note also `tier: 1` — the
number came back with no consent or cost gate, because a scalar summary
exposes no participant-/event-level raw stream.

## 3. Tier 2 — the consent gate, then the trace

A downsampled acceleration trace is more than a scalar — it is the actual
ground motion, sample by sample (decimated). The router blocks the first
call with a **structured consent response** the client must surface; only an
explicit `approve_consent_strong_motion` unblocks the session. Then the same
call returns the trace, which we plot.

In [ ]:
# Attempt 1 — consent not yet granted, so the router short-circuits.
gate = await call(
    "seismic_downsampled",
    {"file_id": "TARZANA.RAW", "interval": 20, "entity_id": ENTITY},
)
print("[1] gate fired :", gate.get("gate"))
print("    user prompt:", gate.get("user_prompt"))

# Approve, then retry.
approval = await call("approve_consent_strong_motion")
print("\n[2]", approval.get("message"))

trace = await call(
    "seismic_downsampled",
    {"file_id": "TARZANA.RAW", "interval": 20, "entity_id": ENTITY},
)
samples = trace["samples"]
step_dt = trace["dt_s"] * trace["interval"]
t = [i * step_dt for i in range(len(samples))]
print(f"\n[3] downsampled to {len(samples)} points (every {trace['interval']}th sample), units = {trace['units']}")

In [ ]:
# Plot the downsampled trace. Defensive import so the notebook still runs
# if matplotlib is absent (the trace summary prints instead).
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 3))
    plt.plot(t, samples, linewidth=0.7)
    plt.axhline(0, color="0.7", linewidth=0.5)
    plt.xlabel("time (s)")
    plt.ylabel("acceleration (g)")
    plt.title("Tarzana CHAN 1 (90 deg) — downsampled acceleration "
              "(raw uncorrected peak (V1) = 1.927 g)")
    plt.tight_layout()
    plt.show()
except ImportError:
    peak = max(abs(x) for x in samples)
    print("(matplotlib not installed — trace summary instead)")
    print(f"  {len(samples)} points, peak |a| in downsampled trace = {peak:.3f} g")

## 4. Vault round-trip — durable, human-readable memory

The reasoning tier persists as plain markdown in an Obsidian-compatible
vault. We write an **evidence note** recording the finding; the file on disk
is the source of truth a human can open and edit.

In [ ]:
await call("vault_capture_moment", {
    "title": "Tarzana 90deg raw PGA = 1.927 g (Northridge 1994)",
    "body": (
        "The raw uncorrected (V1) peak ground acceleration for the Tarzana - "
        "Cedar Hill Nursery CHAN 1 (90 deg) record of the 1994 Northridge "
        "earthquake is 1.927 g, at ~8.35 s. This is the raw uncorrected peak; "
        "the instrument-corrected V2 peak is lower (1.779 g) — see the "
        "reproduction cell. Computed server-side from the COSMOS V1 record; "
        "no raw trace entered the model context."
    ),
    "tags": ["strong-motion", "northridge", "pga", "v1-uncorrected"],
    "entity_id": ENTITY,
})

note_files = sorted(p for p in vault_dir.rglob("*.md") if "moment" in str(p).lower())
note_file = note_files[0] if note_files else next(vault_dir.rglob("*.md"))
print("evidence note written to:")
print(" ", note_file)
print("\n--- markdown source (what a researcher sees in Obsidian) ---")
print(note_file.read_text(encoding="utf-8")[:900])

## 5. The audit log — the reproducibility backbone

Every dispatch appended a row to `audit.db`: when, which tool, what tier,
the outcome, the `entity_id`, and the `scrubber_id` recording which
data-scrubber policy was in force ([ADR 0003](../adr/0003-phi-scrubber-seam.md)).
This is the answer to "what did an analyst see, when, at what tier, on which
record?" — and we assert the scrubber identity is on the row.

In [ ]:
with sqlite3.connect(str(data_dir / "audit.db")) as conn:
    row = conn.execute(
        "SELECT timestamp, domain, tool_name, tier, outcome, entity_id, scrubber_id "
        "FROM audit_log WHERE tool_name = 'seismic_record_summary' "
        "ORDER BY id DESC LIMIT 1"
    ).fetchone()

ts, domain, tool, tier, outcome, entity_id, scrubber_id = row
print(f"timestamp  : {ts}")
print(f"domain     : {domain}")
print(f"tool       : {tool}")
print(f"tier       : {tier}")
print(f"outcome    : {outcome}")
print(f"entity_id  : {entity_id}")
print(f"scrubber_id: {scrubber_id}")

assert scrubber_id is not None and scrubber_id != "", \
    "audit row must carry a scrubber_id (ADR 0003 — 'did we scrub?' is a fact on disk)"
print("\nASSERT scrubber_id present on the audit row: PASS")

## 6. Reproduction — the hard assertion, and the provenance lesson

This is the cell that makes the notebook a *result*, not a demo: it asserts
the computed raw uncorrected peak equals the documented **1.927 g (V1)**. If
the parser or the math ever drifts, this fails loudly.

Then it contrasts that with the **processed V2** value to make the
provenance point explicit.

In [ ]:
raw_pga = summary["pga_g"]
print(f"raw uncorrected peak (V1) : {raw_pga:.4f} g   (computed from the {RECORD_SOURCE})")

assert round(raw_pga, 3) == RAW_PGA_EXPECTED, (
    f"raw uncorrected PGA should be {RAW_PGA_EXPECTED} g, got {raw_pga:.4f} g"
)
print(f"ASSERT raw uncorrected PGA (V1) == {RAW_PGA_EXPECTED} g : PASS")

print()
print(f"processed instrument-corrected peak (V2) : {V2_PGA:.3f} g")
print("    source: ce24436p.zip / TARZANA.V2 — 'PEAK ACCELERATION = 1744.533 CM/SEC/SEC'")
print("            1744.533 cm/s^2 / 980.665 cm/s^2-per-g = 1.779 g")

**The gap (1.927 g → 1.779 g) is the whole point.**

The two numbers describe the *same shaking*, but they are not
interchangeable, and which one is "correct" depends on the question:

- **1.927 g — raw uncorrected (V1).** The maximum of the as-recorded
  accelerogram, straight off the instrument. It still carries the sensor's
  own instrument response and baseline; it is the right number when you want
  *what the instrument actually recorded*.
- **1.779 g — instrument-corrected (V2).** After baseline correction,
  instrument-response removal, and band-pass filtering. It is the right
  number for *engineering ground-motion* use (response-spectrum comparison,
  code checks).

A result that quotes "1.927 g" without saying **V1 / raw uncorrected** is
ambiguous — a reader cannot tell which processing stage produced it. That is
exactly the failure Tailor's audit log and `_meta` provenance stamps exist
to prevent: the number is always traceable to the bytes and the code that
produced it. The pinned SHA-256 in Section 1 closes the loop on *which bytes*.

## Coda — same pipeline, different planet

Nothing above was seismic-specific in the *framework*. To prove it, point the
generic `csv_dir` child at a completely unrelated public dataset — the **Ames
Housing** dataset (2,930 residential sales, De Cock 2011) — and run the same
kind of governed Tier-1 summary. No new child, no new code.

In [ ]:
ames_bytes, ames_prov = fetch_pinned(AMES_URL, AMES_SHA256)
print("PROVENANCE (Ames Housing)")
print("  source URL   :", ames_prov["url"])
print("  SHA-256      :", ames_prov["sha256_expected"])
print("  license      :", AMES_LICENSE)
print("  retrieved at :", ames_prov["retrieved_at"])
print("  status       :", ames_prov["status"])

ames_dir = workdir / "ames"; ames_dir.mkdir()
# AmesHousing.txt is TAB-separated; csv_dir reads comma CSV, so convert.
# (Data-prep glue — the child itself is unchanged.)
if ames_bytes is not None:
    rows = list(csv.reader(io.StringIO(ames_bytes.decode("utf-8")), delimiter="\t"))
    with open(ames_dir / "ames.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)
    print(f"\nAmes: {len(rows[0])} columns, {len(rows) - 1} data rows (UNFILTERED)")
    AMES_AVAILABLE = True
else:
    print("\n(Ames download unavailable — skipping the coda; offline run)")
    AMES_AVAILABLE = False

In [ ]:
if AMES_AVAILABLE:
    from tailor.children.csv_dir import CSVDirectoryChild

    # Same config file, now with a csv_dir block alongside strong_motion.
    (config_dir / "user_config.json").write_text(json.dumps({
        "strong_motion": {"path": str(records_dir)},
        "csv_dir": {
            "path": str(ames_dir),
            "value_columns": {
                "SalePrice": "Sale price (USD)",
                "Gr Liv Area": "Above-grade living area (sq ft)",
            },
        },
    }), encoding="utf-8")

    ames_child = CSVDirectoryChild(config_dir=config_dir, data_dir=data_dir)

    detail = await ames_child.execute("csv_file_detail", {"file_id": "ames.csv"})
    print("csv_file_detail row_count:", detail["row_count"])
    # UNFILTERED: the 5 GrLivArea > 4000 outliers De Cock flags are kept.
    assert detail["row_count"] == AMES_ROWS_EXPECTED, (
        f"Ames is {AMES_ROWS_EXPECTED} rows UNFILTERED; got {detail['row_count']}"
    )
    print(f"ASSERT Ames row_count == {AMES_ROWS_EXPECTED} (UNFILTERED, outliers kept): PASS")

    report = await ames_child.execute("csv_summary_report", {"file_id": "ames.csv"})
    sp = report["column_summaries"]["SalePrice"]
    print(f"\nSalePrice  n={sp['count']}  mean=${sp['mean']:,.0f}  "
          f"min=${sp['min']:,.0f}  max=${sp['max']:,.0f}")
else:
    print("Coda skipped (offline).")

**Same pipeline, earthquake → rent roll, zero new code.** The router, the
tier model, the consent/cost gates, the audit log, and the vault did not
change between a strong-motion accelerogram and a housing table — only the
child that wraps the source did. That is the whole architectural bet: put the
governance in the server, and any data source inherits it.

In [ ]:
router.close()
print("Closed. Workspace was at:", workdir)